In [ ]:
from pathlib import Path
import pandas as pd


def localizar_raiz_projeto():
    """Encontra a raiz do projeto ao executar da raiz ou de uma subpasta."""
    for pasta in (Path.cwd(), *Path.cwd().parents):
        if (pasta / "data").is_dir() and (pasta / "lake").is_dir():
            return pasta

    raise FileNotFoundError(
        "Nao foi possivel localizar a raiz do projeto (pastas 'data' e 'lake')."
    )


def main():
    raiz_projeto = localizar_raiz_projeto()
    caminho_bronze = (
        raiz_projeto
        / "lake"
        / "bronze"
        / "ecommerce"
        / "ecommerce_orders_dirty.csv"
    )
    pasta_silver = raiz_projeto / "lake" / "silver" / "ecommerce"
    caminho_silver = pasta_silver / "ecommerce_orders_clean.csv"

    if not caminho_bronze.is_file():
        raise FileNotFoundError(
            f"Arquivo nao encontrado: {caminho_bronze}. "
            "Execute primeiro o notebook de ingestao."
        )

    dfBronze = pd.read_csv(caminho_bronze)
    colunas_origem = list(dfBronze.columns)
    df = dfBronze.copy()
    linhas_origem = len(df)

    # Normaliza campos textuais e categorias conhecidas.
    for coluna in df.select_dtypes(include=["object", "string"]).columns:
        df[coluna] = df[coluna].astype("string").str.strip()

    df["Product"] = df["Product"].astype("string").str.title()
    df["OrderStatus"] = df["OrderStatus"].astype("string").str.title()
    pagamento = df["PaymentMethod"].astype("string").str.casefold()
    mapa_pagamentos = {
        "online": "Online",
        "on-line": "Online",
        "credit": "Credit Card",
        "credit card": "Credit Card",
        "debit_card": "Debit Card",
        "debit card": "Debit Card",
        "gift card": "Gift Card",
        "cash": "Cash",
    }
    df["PaymentMethod"] = pagamento.map(mapa_pagamentos).fillna(df["PaymentMethod"])

    # Converte tipos antes de deduplicar; datas invalidas viram nulas.
    data_original = df["Date"].copy()
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    quantidade_original = pd.to_numeric(df["Quantity"], errors="coerce").astype("float64")
    df["Quantity"] = quantidade_original
    preco_original = df["UnitPrice"].astype("string")
    preco_numerico = pd.to_numeric(
        preco_original
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce",
    )
    df["UnitPrice"] = preco_numerico
    df["TotalPrice"] = pd.to_numeric(df["TotalPrice"], errors="coerce")

    # Remove duplicatas completas depois da padronizacao, preservando datas
    # invalidas distintas para que nao sejam fundidas por causa do NaT.
    chave = df[colunas_origem].copy()
    chave["Date"] = df["Date"].astype("object")
    datas_invalidas = df["Date"].isna()
    chave.loc[datas_invalidas, "Date"] = data_original.loc[datas_invalidas].astype("object")
    duplicatas_padronizadas = chave.duplicated(keep="first")
    removidas_padronizacao = int(duplicatas_padronizadas.sum())
    df = df.loc[~duplicatas_padronizadas].copy()

    # Valores ausentes e sentinelas usam as regras definidas para este projeto.
    df["DateWasInvalid"] = df["Date"].isna()
    df["QuantityWasCorrected"] = df["Quantity"].lt(0)
    df.loc[df["Quantity"].lt(0), "Quantity"] = df.loc[
        df["Quantity"].lt(0), "Quantity"
    ].abs()

    quantidades_validas = df.loc[
        df["Quantity"].gt(0) & df["Quantity"].ne(9999), "Quantity"
    ]
    if quantidades_validas.empty:
        raise ValueError("Nao ha quantidades validas para calcular a media.")
    media_quantidade = int(round(quantidades_validas.mean()))
    mascara_9999 = df["Quantity"].eq(9999)
    df["QuantityWasImputed"] = mascara_9999
    df.loc[mascara_9999, "Quantity"] = media_quantidade

    df["UnitPriceWasImputed"] = df["UnitPrice"].isna()
    df["UnitPrice"] = df["UnitPrice"].fillna(0)

    # Guarda o total informado e recalcula o total canonico para manter
    # consistencia com as quantidades e precos limpos.
    df["TotalPrice_Source"] = df["TotalPrice"]
    df["TotalPrice"] = (df["Quantity"] * df["UnitPrice"]).round(2)
    df["Quantity"] = df["Quantity"].astype("int64")

    # Consolida duplicatas que ficaram identicas apos as correcoes.
    colunas_chave = [coluna for coluna in df.columns if coluna != "TotalPrice_Source"]
    duplicatas_corrigidas = df.duplicated(subset=colunas_chave, keep="first")
    removidas_correcao = int(duplicatas_corrigidas.sum())
    dfSilver = df.loc[~duplicatas_corrigidas].copy()

    # Validacoes da camada Silver segundo as regras solicitadas.
    if dfSilver["Quantity"].le(0).any() or dfSilver["Quantity"].eq(9999).any():
        raise ValueError("A Silver ainda contem quantidade nao tratada.")
    if dfSilver["UnitPrice"].isna().any():
        raise ValueError("A Silver ainda contem preco unitario ausente.")
    if (dfSilver["TotalPrice"] - dfSilver["Quantity"] * dfSilver["UnitPrice"]).abs().gt(0.01).any():
        raise ValueError("A Silver ainda contem divergencia entre quantidade, preco e total.")
    if dfSilver.duplicated(subset=colunas_chave).any():
        raise ValueError("A Silver ainda contem duplicatas completas.")

    pasta_silver.mkdir(parents=True, exist_ok=True)
    dfSilver.to_csv(caminho_silver, index=False)

    print(f"Linhas na Bronze: {linhas_origem}")
    print(f"Duplicatas removidas apos padronizacao: {removidas_padronizacao}")
    print(f"Duplicatas removidas apos imputacao/correcao: {removidas_correcao}")
    print(f"Linhas gravadas na Silver: {len(dfSilver)}")
    print(f"Media valida de Quantity arredondada: {media_quantidade}")
    print(f"Datas invalidas mantidas como nulas: {int(dfSilver['DateWasInvalid'].sum())}")
    print(f"Quantidades 9999 substituidas: {int(dfSilver['QuantityWasImputed'].sum())}")
    print(f"Precos ausentes substituidos por zero: {int(dfSilver['UnitPriceWasImputed'].sum())}")
    print(f"TotalPrice original preservado em TotalPrice_Source: {int(dfSilver['TotalPrice_Source'].notna().sum())} valores")
    print(f"Arquivo Silver: {caminho_silver}")


if __name__ == "__main__":
    main()


In [4]:
def main():
    raiz_projeto = localizar_raiz_projeto()
    caminho_bronze = (
        raiz_projeto
        / "lake"
        / "bronze"
        / "ecommerce"
        / "ecommerce_orders_dirty.csv"
    )

    if not caminho_bronze.is_file():
        raise FileNotFoundError(
            f"Arquivo nao encontrado: {caminho_bronze}. "
            "Execute primeiro o notebook de ingestao."
        )

    dfBronze = pd.read_csv(caminho_bronze)

    
    # 1. Investigar as duplicatas
    print("\nQuantidade de linhas:")
    print(len(dfBronze))

    print("\nQuantidade de linhas após remover duplicatas exatas (simulação):")
    print(len(dfBronze.drop_duplicates()))

    print("\nExemplos de linhas duplicadas:")
    print(dfBronze[dfBronze.duplicated(keep=False)].head(10))


    # 2. Investigar UnitPrice
    print("\nExemplos de valores de UnitPrice:")
    print(dfBronze["UnitPrice"].value_counts(dropna=False).head(20))


    # 3. Investigar valores de categorias
    for coluna in ["PaymentMethod", "OrderStatus", "Product"]:
        print(f"\nValores distintos em {coluna}:")
        print(dfBronze[coluna].value_counts(dropna=False))


    # 4. Investigar quantidades negativas ou iguais a zero
    print("\nQuantity menor ou igual a zero:")
    print(dfBronze[dfBronze["Quantity"] <= 0][
        ["OrderID", "Quantity", "UnitPrice", "OrderStatus", "TotalPrice"]
    ].head(20))


    # 5. Investigar consistência entre quantidade,
    # preço unitário e preço total
    dfBronze["UnitPrice_numerico"] = pd.to_numeric(
        dfBronze["UnitPrice"]
        .astype("string")
        .str.replace("$", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

    dfBronze["Total_calculado"] = (
        dfBronze["Quantity"] * dfBronze["UnitPrice_numerico"]
    )

    dfBronze["Diferenca_total"] = (
        dfBronze["TotalPrice"] - dfBronze["Total_calculado"]
    )

    print("\nExemplos de divergência no total:")
    print(
        dfBronze[
            dfBronze["Diferenca_total"].abs().gt(0.01)
            & dfBronze["Diferenca_total"].notna()
        ][
            ["OrderID", "Quantity", "UnitPrice",
            "TotalPrice", "Total_calculado"]
        ].head(20)
    )

    print(
        "\nQuantidade de preços unitários que não puderam "
        "ser convertidos:",
        dfBronze["UnitPrice_numerico"].isna().sum()
    )

if __name__ == "__main__":
    main()


Quantidade de linhas:
12500

Quantidade de linhas após remover duplicatas exatas (simulação):
6567

Exemplos de linhas duplicadas:
      OrderID                 Date CustomerID     Product  Quantity UnitPrice  \
1   ORD200001  2024-08-23 00:00:00     C75739       Phone         2    151.35   
3   ORD200003  2023-10-15 00:00:00     C33540       Chair         1    273.19   
5   ORD200005  2023-10-23 00:00:00     C37249      phone          2   $245.86   
6   ORD200006  2025-06-17 00:00:00     C83492     laptop          1    664.42   
7   ORD200007  2023-12-05 00:00:00     C41460     Monitor         5    149.55   
8   ORD200008  2025-04-02 00:00:00     C26817       Phone         2    134.28   
9   ORD200009  2023-11-21 00:00:00     C31946        Desk         4    509.38   
10  ORD200010  2023-12-29 00:00:00     C43443      Tablet         5    625.97   
12  ORD200012  2024-10-15 00:00:00     C38785    monitor          2     180.5   
13  ORD200013  2023-08-30 00:00:00     C88348      Laptop 

In [8]:

def main():
    raiz_projeto = localizar_raiz_projeto()
    caminho_bronze = (
        raiz_projeto
        / "lake"
        / "bronze"
        / "ecommerce"
        / "ecommerce_orders_dirty.csv"
    )

    if not caminho_bronze.is_file():
        raise FileNotFoundError(
            f"Arquivo nao encontrado: {caminho_bronze}. "
            "Execute primeiro o notebook de ingestao."
        )

    dfBronze = pd.read_csv(caminho_bronze)


    # Investigar identificadores de pedidos
    print("\nTotal de OrderID distintos:")
    print(dfBronze["OrderID"].nunique())

    print("\nOrderID repetidos:")
    ids_repetidos = dfBronze[
        dfBronze.duplicated(subset=["OrderID"], keep=False)
    ].sort_values("OrderID")

    print("Quantidade de linhas com OrderID repetido:", len(ids_repetidos))
    print(ids_repetidos.head(20))

    # Verificar se um mesmo OrderID possui dados diferentes
    print("\nQuantidade de OrderID com mais de uma versão:")
    variacoes_por_pedido = (
        dfBronze.groupby("OrderID")
        .nunique()
    )

    colunas_com_variacao = variacoes_por_pedido[
        variacoes_por_pedido.gt(1).any(axis=1)
    ]

    print(len(colunas_com_variacao))
    print(colunas_com_variacao.head(20))

    # Investigar quantidades suspeitas
    print("\nDistribuição das quantidades:")
    print(dfBronze["Quantity"].describe())

    print("\nQuantidade de registros com Quantity negativa:")
    print((dfBronze["Quantity"] < 0).sum())

    print("\nQuantidade de registros com Quantity igual a zero:")
    print((dfBronze["Quantity"] == 0).sum())

    print("\nQuantidade de registros com Quantity igual a 9999:")
    print((dfBronze["Quantity"] == 9999).sum())

    
    # 1. Distribuição da quantidade de linhas por OrderID
    print("\nQuantidade de linhas por pedido:")
    print(dfBronze["OrderID"].value_counts().value_counts().sort_index())

    # 2. Comparar linhas totalmente duplicadas
    print("\nLinhas totalmente duplicadas:")
    print(dfBronze.duplicated().sum())

    # 3. Verificar diferenças de formatação em Product
    produto_normalizado = dfBronze["Product"].astype("string").str.strip().str.lower()

    print("\nProdutos antes da normalização:")
    print(dfBronze["Product"].nunique())

    print("\nProdutos após a normalização:")
    print(produto_normalizado.nunique())

    # 4. Verificar diferenças de formatação em PaymentMethod
    pagamento_normalizado = (
        dfBronze["PaymentMethod"]
        .astype("string")
        .str.strip()
        .str.lower()
        .replace({
            "on-line": "online",
            "credit": "credit card",
            "debit_card": "debit card"
        })
    )

    print("\nFormas de pagamento antes da normalização:")
    print(dfBronze["PaymentMethod"].nunique())

    print("\nFormas de pagamento após a normalização:")
    print(pagamento_normalizado.nunique())

if __name__ == "__main__":
    main()


Total de OrderID distintos:
1200

OrderID repetidos:
Quantidade de linhas com OrderID repetido: 12500
         OrderID                 Date CustomerID     Product  Quantity  \
0      ORD200000  2023-01-04 00:00:00     C72649     Monitor        -5   
8400   ORD200000  2023-01-04 00:00:00     C72649     Monitor         5   
1200   ORD200000  2023-01-04 00:00:00     C72649    monitor          5   
9600   ORD200000  2023-01-04 00:00:00     C72649    monitor          5   
7200   ORD200000  2023-01-04 00:00:00     C72649     Monitor         5   
10800  ORD200000  2023-01-04 00:00:00     C72649     Monitor         5   
2400   ORD200000  2023-04-01 00:00:00     C72649     Monitor         5   
3600   ORD200000  2023-01-04 00:00:00     C72649    monitor          5   
4800   ORD200000  2023-01-04 00:00:00     C72649     Monitor         5   
6000   ORD200000  2023-01-04 00:00:00     C72649     Monitor         5   
7201   ORD200001  2024-08-23 00:00:00     C75739       Phone         2   
1201   OR